# TEM Ingestion Demo

Ingests all `*_MITO_measurements.txt` files produced by ImageJ/Fiji into a single AnnData object.

**Steps:**
1. (Optional) Re-ingest raw `.txt` files from scratch with `ingest_tem_folder()`
2. Load the saved `.h5ad` and inspect the result
3. Preview `.obs`, `.var`, and `.X`

---
**Google Colab setup** — mount your Drive first:
```python
from google.colab import drive
drive.mount('/content/drive')
TEM_DATA_DIR  = '/content/drive/MyDrive/mito-marker/data/raw/tem/'
TEM_H5AD_PATH = '/content/drive/MyDrive/mito-marker/data/processed/tem.h5ad'
```

**GitHub Codespaces setup** — point to local directories:
```python
TEM_DATA_DIR  = '../data/raw/tem/'
TEM_H5AD_PATH = '../data/processed/tem.h5ad'
```

## Cell 1 — (Optional) Ingest raw `.txt` files

Run this cell once to process all `*_MITO_measurements.txt` files in your raw data folder.
The result is saved as an `.h5ad` file. Skip this cell if `.h5ad` already exists.

Set `append_to_existing=True` to add new files to an existing `.h5ad` without re-processing old ones.

In [ ]:
from mito_marker import ingest_tem_folder

# --- Adjust these paths for your environment ---
TEM_DATA_DIR  = '../data/raw/tem/'           # folder containing *_MITO_measurements.txt
TEM_H5AD_PATH = '../data/processed/tem.h5ad' # output file

# Google Colab example:
# from google.colab import drive
# drive.mount('/content/drive')
# TEM_DATA_DIR  = '/content/drive/MyDrive/mito-marker/data/raw/tem/'
# TEM_H5AD_PATH = '/content/drive/MyDrive/mito-marker/data/processed/tem.h5ad'

tem_anndata = ingest_tem_folder(
    data_directory_path=TEM_DATA_DIR,
    output_file_path=TEM_H5AD_PATH,
    append_to_existing=False,  # set True to add new files to an existing .h5ad
)

## Cell 2 — Load the saved AnnData

If you already ran Cell 1 and just want to reload the saved file, run this cell directly.

In [ ]:
import anndata

TEM_H5AD_PATH = '../data/processed/tem.h5ad'
# Google Colab: TEM_H5AD_PATH = '/content/drive/MyDrive/mito-marker/data/processed/tem.h5ad'

print(f"Loading: {TEM_H5AD_PATH}")
tem_anndata = anndata.read_h5ad(TEM_H5AD_PATH)
print(f"Loaded — {tem_anndata.n_obs:,} mitochondria × {tem_anndata.n_vars} features")

## Cell 3 — Inspect the result

Quick overview of what was ingested: per-group mitochondrion counts,
feature names, and a sanity check on `.X`.

In [ ]:
import numpy as np

print("=" * 50)
print("AnnData overview")
print("=" * 50)
print(f"Total mitochondria : {tem_anndata.n_obs:,}")
print(f"Morphological features : {tem_anndata.n_vars}")

print("\n--- Mitochondria per group ---")
print(tem_anndata.obs.groupby(["specie", "age_group", "subject_ID"]).size().to_string())

print("\n--- .obs head ---")
print(tem_anndata.obs[["specie", "age_group", "subject_ID", "Image_Name", "source_filename"]].head(5).to_string())

print("\n--- .var (feature names) ---")
print(list(tem_anndata.var_names))

print("\n--- .X sanity check ---")
print(f"dtype  : {tem_anndata.X.dtype}")
print(f"shape  : {tem_anndata.X.shape}")
print(f"NaN    : {int(np.isnan(tem_anndata.X).sum())}")
print(f"min    : {float(tem_anndata.X.min()):.4f}")
print(f"max    : {float(tem_anndata.X.max()):.4f}")
print(f"mean   : {float(tem_anndata.X.mean()):.4f}")

## Cell 4 — (Quick test) Ingest the bundled fixture files

Runs ingestion on the three small fixture files committed in `tests/fixtures/`.
Useful to verify the pipeline works without any external data.

In [ ]:
from mito_marker import ingest_tem_folder

tem_fixture_anndata = ingest_tem_folder(
    data_directory_path='../tests/fixtures/',
    output_file_path='../data/processed/tem_fixtures.h5ad',
)

print(f"\nFixture result: {tem_fixture_anndata.n_obs} mitochondria × {tem_fixture_anndata.n_vars} features")
print(tem_fixture_anndata.obs[["specie", "age_group", "subject_ID", "source_filename"]].to_string())